# Step 3: LLM-as-Judge Evaluation

**Input:** `ads_llm.json` + `ad_generation_input.json`  
**Output:** `scores_pointwise.csv`, `scores_pairwise.csv`, `results_summary.txt`

**Generator:** GPT-5.4 (OpenAI)  
**Judge:** DeepSeek-V4-Flash (DeepSeek) -- cross-company, avoids self-enhancement bias

**Two evaluation modes:**
- **Part A Pointwise** (supplementary): each ad scored independently on 4 dimensions (ELM + Rossiter-Percy)
- **Part B Pairwise** (primary): A vs B shown simultaneously; judge picks the winner; each pair run twice to control position bias

Total calls: 18 pointwise + 18 pairwise = 36

## 1. Install & Import

In [1]:
import json
import re
import time
import random
import pandas as pd
from openai import OpenAI

## 2. Configuration

Get a free API key at [platform.deepseek.com](https://platform.deepseek.com).  
Judge: `deepseek-v4-flash` -- different company and architecture from GPT-5.4, mitigates self-enhancement bias.

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

# Load API keys from .env
load_dotenv()
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
if not DEEPSEEK_API_KEY:
    raise RuntimeError(
        "DEEPSEEK_API_KEY not found."
    )

JUDGE_MODEL = "deepseek-v4-flash"
GEN_MODEL   = "gpt-5.4"

client = OpenAI(
    api_key  = DEEPSEEK_API_KEY,
    base_url = "https://api.deepseek.com/v1",
)
print(f"Generator : {GEN_MODEL}")
print(f"Judge     : {JUDGE_MODEL}")

## 3. Load Data

In [3]:
with open('ads_llm.json', 'r', encoding='utf-8') as f:
    ads_data = json.load(f)

with open('ad_generation_input.json', 'r', encoding='utf-8') as f:
    input_data = json.load(f)

input_lookup = {p['asin']: p for p in input_data}

n_ads = sum(len(p['ads']) * 2 for p in ads_data)
print(f'Products : {len(ads_data)}')
print(f'Total ads: {n_ads}')

Products : 3
Total ads: 18


## 4. Shared: API Call Helper

In [4]:
def call_judge(prompt, max_tokens=3000, max_retries=3):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=JUDGE_MODEL,
                messages=[{'role': 'user', 'content': prompt}],
                temperature=0.2,
                max_tokens=max_tokens,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            wait = 2 ** attempt
            print(f'  [Attempt {attempt+1} failed: {e} -- retrying in {wait}s]')
            time.sleep(wait)
    return None

print('call_judge defined.')

call_judge defined.


---
## Part A: Pointwise Evaluation (supplementary)

Each ad is scored independently on 4 dimensions grounded in ELM and Rossiter-Percy Grid.  
Results serve as supplementary evidence alongside the pairwise comparison.

Dimensions:
- **Emotional Resonance** (ELM peripheral / Rossiter-Percy): treats excitement (A) and reassurance (B) as equally valid
- **Argument Strength** (ELM central route): specificity and logical quality of purchase rationale
- **Specificity**: product-specific language vs generic marketing filler
- **Purchase Motivation** (AIDA): calibrated for 2-3 sentence format; score 3 = neutral

In [5]:
def build_pointwise_prompt(product_title, description, features, topic,
                            positive_reviews, negative_reviews, ad_text):
    features_text = '\n'.join(f'- {f}' for f in features[:5]) if features else 'Not provided'
    pos_text = '\n'.join(f'  [{i+1}] {r[:280]}' for i, r in enumerate(positive_reviews[:5]))
    neg_text = '\n'.join(f'  [{i+1}] {r[:280]}' for i, r in enumerate(negative_reviews[:5]))

    prompt = (
        'You are an expert advertising evaluator. Score the advertisement below on four dimensions.\n'
        'Be critical and use the FULL 1-5 scale.\n'
        '\n'
        'CALIBRATION: This is a short-form ad (2-3 sentences). Do NOT penalise it for lacking elements\n'
        'of longer ads. Judge how well it achieves its goal within its format.\n'
        'Score 3 = truly mediocre. Score 4 = well-executed short ad. Score 5 = outstanding.\n'
        'Scores 1-2 for clear failures.\n'
        '\n'
        '===========================================\n'
        f'PRODUCT: {product_title}\n'
        f'Feature advertised: {topic}\n'
        f'Description: {description[:500]}\n'
        f'Key features:\n{features_text}\n'
        '\n'
        f'POSITIVE REVIEWS about {topic}:\n{pos_text}\n'
        f'\nNEGATIVE REVIEWS about {topic}:\n{neg_text}\n'
        '\n'
        'ADVERTISEMENT TO EVALUATE:\n'
        f'{ad_text}\n'
        '\n'
        '===========================================\n'
        'SCORING RUBRICS\n'
        '\n'
        '1. EMOTIONAL RESONANCE (1-5)  [ELM peripheral / Rossiter-Percy]\n'
        '   Two equally valid paths -- judge intensity, not type:\n'
        '   Positive path: excitement, aspiration, delight\n'
        '   Reassurance path: trust, relief, confidence\n'
        '   5 = Strong vivid emotion clearly evoked\n'
        '   4 = Clear emotional tone that resonates\n'
        '   3 = Weak/generic emotional cues; flat\n'
        '   2 = Minimal emotion; reads like a spec list\n'
        '   1 = No emotion or off-putting\n'
        '\n'
        '2. ARGUMENT STRENGTH (1-5)  [ELM central route]\n'
        '   5 = Specific concrete claims with clear reasoning\n'
        '   4 = Mostly specific; core logic is sound\n'
        '   3 = Mix of specific and generic\n'
        '   2 = Mostly generic; could describe any product\n'
        '   1 = Pure marketing fluff\n'
        '\n'
        '3. SPECIFICITY (1-5)  [Concreteness of language]\n'
        '   Specific: product names, named features, mechanisms, concrete outcomes\n'
        '   Generic: "great quality", "easy to use", "perfect for beginners"\n'
        '   5 = Precise and product-specific throughout\n'
        '   4 = Mostly specific; one generic phrase tolerated\n'
        '   3 = Balanced mix\n'
        '   2 = Mostly generic\n'
        '   1 = Entirely generic\n'
        '\n'
        '4. PURCHASE MOTIVATION (1-5)  [AIDA desire + action]\n'
        '   For a 2-3 sentence ad, how effectively does it create desire?\n'
        '   5 = Highly compelling; hard to dismiss\n'
        '   4 = Noticeably motivating; inclines toward purchase\n'
        '   3 = Neutral; neither motivates nor discourages\n'
        '   2 = Mildly discouraging; claims feel hollow\n'
        '   1 = Actively off-putting\n'
        '\n'
        '===========================================\n'
        'OUTPUT FORMAT\n'
        '\n'
        'REASONING\n'
        'Emotional Resonance: <2-3 sentences>\n'
        'Argument Strength: <2-3 sentences>\n'
        'Specificity: <2-3 sentences>\n'
        'Purchase Motivation: <2-3 sentences>\n'
        '\n'
        'JSON\n'
        '```json\n'
        '{\n'
        '  "emotional_resonance": <integer 1-5>,\n'
        '  "argument_strength": <integer 1-5>,\n'
        '  "specificity": <integer 1-5>,\n'
        '  "purchase_motivation": <integer 1-5>\n'
        '}\n'
        '```'
    )
    return prompt

print('build_pointwise_prompt defined.')

build_pointwise_prompt defined.


In [6]:
def parse_pointwise_response(raw_text):
    if not raw_text:
        return None

    reasoning = {}
    for dim in ['Emotional Resonance', 'Argument Strength', 'Specificity', 'Purchase Motivation']:
        pattern = rf'{dim}:\s*(.+?)(?=\n[A-Z]|\nJSON|```|$)'
        match = re.search(pattern, raw_text, re.DOTALL | re.IGNORECASE)
        if match:
            reasoning[dim.lower().replace(' ', '_')] = match.group(1).strip()

    json_obj = None
    match = re.search(r'```(?:json)?\s*(\{.*?\})\s*```', raw_text, re.DOTALL)
    if match:
        try:
            json_obj = json.loads(match.group(1))
        except json.JSONDecodeError:
            pass

    if json_obj is None:
        match = re.search(r'\{[^{}]*"emotional_resonance"[^{}]*\}', raw_text, re.DOTALL)
        if match:
            try:
                json_obj = json.loads(match.group(0))
            except json.JSONDecodeError:
                pass

    if json_obj is None:
        print('  [WARNING: could not parse JSON]')
        print('  Raw:', raw_text[:200])
        return None

    json_obj['reasoning'] = reasoning
    return json_obj

print('parse_pointwise_response defined.')

parse_pointwise_response defined.


### A1. Score All 18 Ads (Pointwise)

In [7]:
pointwise_scores = []
pointwise_raw    = []

for product in ads_data:
    asin = product['asin']
    inp  = input_lookup.get(asin, {})
    print(f"\n{'='*60}")
    print(f"{asin} | {product['product_title'][:50]}")

    for topic, ads in product['ads'].items():
        ti  = inp.get('topics', {}).get(topic, {})
        pos = ti.get('positive_reviews', [])
        neg = ti.get('negative_reviews', [])

        for strat_label, ad_text in [('A', ads['strategy_a']), ('B', ads['strategy_b'])]:
            print(f"  [{topic[:20]}] Strategy {strat_label} ...", end=' ')

            prompt = build_pointwise_prompt(
                product_title    = product['product_title'],
                description      = inp.get('description', ''),
                features         = inp.get('features', []),
                topic            = topic,
                positive_reviews = pos,
                negative_reviews = neg,
                ad_text          = ad_text,
            )
            raw    = call_judge(prompt)
            parsed = parse_pointwise_response(raw)

            pointwise_raw.append({
                'asin': asin, 'topic': topic, 'strategy': strat_label,
                'ad_text': ad_text, 'raw': raw,
            })

            if parsed:
                row = {
                    'asin': asin, 'product_title': product['product_title'],
                    'brand': product['brand'], 'topic': topic,
                    'strategy': strat_label, 'ad_text': ad_text,
                    'emotional_resonance': parsed.get('emotional_resonance'),
                    'argument_strength':   parsed.get('argument_strength'),
                    'specificity':         parsed.get('specificity'),
                    'purchase_motivation': parsed.get('purchase_motivation'),
                    'reasoning_er': parsed.get('reasoning', {}).get('emotional_resonance', ''),
                    'reasoning_as': parsed.get('reasoning', {}).get('argument_strength', ''),
                    'reasoning_sp': parsed.get('reasoning', {}).get('specificity', ''),
                    'reasoning_pm': parsed.get('reasoning', {}).get('purchase_motivation', ''),
                }
                pointwise_scores.append(row)
                er = parsed.get('emotional_resonance','?')
                a_ = parsed.get('argument_strength','?')
                sp = parsed.get('specificity','?')
                pm = parsed.get('purchase_motivation','?')
                print(f'ER={er} AS={a_} SP={sp} PM={pm}')
            else:
                print('FAILED')

            time.sleep(1)

print(f'\nPointwise complete. Rows: {len(pointwise_scores)} / 18')


B002RXXOX8 | Davison Guitars Full Size Electric Guitar with 10-
  [Tuning stability] Strategy A ... ER=2 AS=2 SP=3 PM=2
  [Tuning stability] Strategy B ... ER=4 AS=4 SP=3 PM=4
  [Fret / neck setup] Strategy A ... ER=3 AS=2 SP=2 PM=3
  [Fret / neck setup] Strategy B ... ER=4 AS=4 SP=4 PM=4
  [String quality] Strategy A ... ER=3 AS=2 SP=1 PM=3
  [String quality] Strategy B ... ER=3 AS=2 SP=2 PM=3

B006CYVD5E | Directly Cheap 6 String Acoustic Guitar Pack, Righ
  [Customer service / r] Strategy A ... ER=2 AS=2 SP=2 PM=2
  [Customer service / r] Strategy B ... ER=3 AS=2 SP=2 PM=3
  [String quality] Strategy A ... ER=3 AS=3 SP=3 PM=3
  [String quality] Strategy B ... ER=2 AS=2 SP=2 PM=3
  [Tuning stability] Strategy A ... ER=3 AS=4 SP=4 PM=4
  [Tuning stability] Strategy B ... ER=3 AS=3 SP=3 PM=3

B002X49732 | Crescent MG38-CF 38" Acoustic Guitar Starter Packa
  [Customer service / r] Strategy A ... ER=3 AS=2 SP=2 PM=3
  [Customer service / r] Strategy B ... ER=4 AS=4 SP=4 PM=4
  [String q

### A2. Save & Analyse Pointwise

In [8]:
pw_df = pd.DataFrame(pointwise_scores)
pw_df.to_csv('scores_pointwise.csv', index=False, encoding='utf-8')

with open('scores_pointwise_raw.json', 'w', encoding='utf-8') as f:
    json.dump(pointwise_raw, f, indent=2, ensure_ascii=False)

dims = ['emotional_resonance', 'argument_strength', 'specificity', 'purchase_motivation']
pw_df['composite'] = pw_df[dims].mean(axis=1)
print(f'Saved scores_pointwise.csv  ({len(pw_df)} rows)')
print(f'Null counts:\n{pw_df[dims].isnull().sum()}')
pw_df[['asin','topic','strategy'] + dims + ['composite']].head(6)

Saved scores_pointwise.csv  (18 rows)
Null counts:
emotional_resonance    0
argument_strength      0
specificity            0
purchase_motivation    0
dtype: int64


,asin,topic,strategy,emotional_resonance,argument_strength,specificity,purchase_motivation,composite
0,B002RXXOX8,Tuning stability,A,2,2,3,2,2.25
1,B002RXXOX8,Tuning stability,B,4,4,3,4,3.75
2,B002RXXOX8,Fret / neck setup,A,3,2,2,3,2.50
3,B002RXXOX8,Fret / neck setup,B,4,4,4,4,4.00
4,B002RXXOX8,String quality,A,3,2,1,3,2.25
5,B002RXXOX8,String quality,B,3,2,2,3,2.50


In [9]:
pw_df = pd.read_csv('scores_pointwise.csv')
dims  = ['emotional_resonance', 'argument_strength', 'specificity', 'purchase_motivation']
pw_df['composite'] = pw_df[dims].mean(axis=1)

print('=== Overall: Strategy A vs B (Pointwise) ===')
overall = pw_df.groupby('strategy')[dims + ['composite']].mean().round(2)
print(overall)

if 'A' in overall.index and 'B' in overall.index:
    diff = (overall.loc['A'] - overall.loc['B']).round(2)
    print(f'\nDifference (A - B):')
    print(diff)
    winner = 'Strategy A' if diff['composite'] > 0 else 'Strategy B'
    print(f'\nPointwise composite winner: {winner}  (Delta={abs(diff["composite"]):.2f})')

print('\n=== Per Dimension ===')
for dim in dims:
    sub = pw_df.groupby('strategy')[dim].mean().round(2)
    w   = 'A' if sub.get('A',0) > sub.get('B',0) else ('B' if sub.get('B',0) > sub.get('A',0) else 'tie')
    print(f'  {dim:<25}  A={sub.get("A","?")}  B={sub.get("B","?")}  -> {w}')

print('\n=== By Topic ===')
for topic in pw_df['topic'].unique():
    sub = pw_df[pw_df['topic']==topic].groupby('strategy')['composite'].mean().round(2)
    w   = 'A' if sub.get('A',0) > sub.get('B',0) else ('B' if sub.get('B',0) > sub.get('A',0) else 'tie')
    delta = abs(sub.get('A',0) - sub.get('B',0))
    print(f'  {topic:<35} -> Strategy {w}  (Delta={delta:.2f})')

print('\n=== By Product ===')
for asin in pw_df['asin'].unique():
    sub   = pw_df[pw_df['asin']==asin]
    title = sub['product_title'].iloc[0][:45]
    means = sub.groupby('strategy')['composite'].mean().round(2)
    w     = 'A' if means.get('A',0) > means.get('B',0) else ('B' if means.get('B',0) > means.get('A',0) else 'tie')
    delta = abs(means.get('A',0) - means.get('B',0))
    print(f'  {title:<45} -> Strategy {w}  (Delta={delta:.2f})')

=== Overall: Strategy A vs B (Pointwise) ===
          emotional_resonance  argument_strength  specificity  \
strategy                                                        
A                        2.78               2.56         2.56   
B                        3.33               3.00         3.00   

          purchase_motivation  composite  
strategy                                  
A                        2.89       2.69  
B                        3.33       3.17  

Difference (A - B):
emotional_resonance   -0.55
argument_strength     -0.44
specificity           -0.44
purchase_motivation   -0.44
composite             -0.48
dtype: float64

Pointwise composite winner: Strategy B  (Delta=0.48)

=== Per Dimension ===
  emotional_resonance        A=2.78  B=3.33  -> B
  argument_strength          A=2.56  B=3.0  -> B
  specificity                A=2.56  B=3.0  -> B
  purchase_motivation        A=2.89  B=3.33  -> B

=== By Topic ===
  Tuning stability                    -> Strategy B  

---
## Part B: Pairwise Evaluation (primary)

Judge sees **both ads simultaneously** and picks the more persuasive one.  
Each pair is evaluated **twice** (A-first, B-first) to control for position bias.  
Final winner per pair: the strategy that wins both orderings, or TIE if they split.

Total calls: 9 pairs x 2 orderings = **18 calls**

In [10]:
def build_pairwise_prompt(product_title, description, features, topic,
                           positive_reviews, negative_reviews, ad1_text, ad2_text):
    features_text = '\n'.join(f'- {f}' for f in features[:5]) if features else 'Not provided'
    pos_text = '\n'.join(f'  [{i+1}] {r[:250]}' for i, r in enumerate(positive_reviews[:5]))
    neg_text = '\n'.join(f'  [{i+1}] {r[:250]}' for i, r in enumerate(negative_reviews[:5]))

    prompt = (
        'You are an expert advertising evaluator. Two short advertisements for the same product\n'
        'and feature are shown below. Decide which is MORE PERSUASIVE to a consumer.\n'
        '\n'
        '===========================================\n'
        f'PRODUCT: {product_title}\n'
        f'Feature advertised: {topic}\n'
        f'Description: {description[:500]}\n'
        f'Key features:\n{features_text}\n'
        '\n'
        f'POSITIVE REVIEWS about {topic}:\n{pos_text}\n'
        f'\nNEGATIVE REVIEWS about {topic}:\n{neg_text}\n'
        '\n'
        '===========================================\n'
        'AD 1:\n'
        f'{ad1_text}\n'
        '\n'
        'AD 2:\n'
        f'{ad2_text}\n'
        '\n'
        '===========================================\n'
        'EVALUATION CRITERIA\n'
        'Compare the two ads holistically. Consider:\n'
        '- Emotional impact: which evokes a stronger, more appropriate emotion?\n'
        '- Argument quality: which gives more specific, convincing reasons to buy?\n'
        '- Persuasive pull: which is more likely to move a consumer toward purchase?\n'
        '\n'
        'Do NOT favour length -- a shorter, sharper ad can outperform a longer, vague one.\n'
        'Do NOT penalise either ad for product flaws visible in the reviews.\n'
        '\n'
        'OUTPUT FORMAT\n'
        'First compare the two ads (3-5 sentences of reasoning).\n'
        'Then state your verdict on a new line.\n'
        '\n'
        'REASONING: <your comparative analysis>\n'
        'WINNER: <1 or 2 or TIE>'
    )
    return prompt

print('build_pairwise_prompt defined.')

build_pairwise_prompt defined.


In [11]:
def parse_pairwise_response(raw_text):
    if not raw_text:
        return None, ''

    reasoning = ''
    m = re.search(r'REASONING:\s*(.+?)(?=\nWINNER:|$)', raw_text, re.DOTALL | re.IGNORECASE)
    if m:
        reasoning = m.group(1).strip()

    winner_num = None
    m = re.search(r'WINNER:\s*(1|2|TIE)', raw_text, re.IGNORECASE)
    if m:
        val = m.group(1).upper()
        winner_num = 'TIE' if val == 'TIE' else int(val)

    if winner_num is None:
        tail = raw_text[-400:]
        if re.search(r'\bAd\s*1\b', tail, re.IGNORECASE) and not re.search(r'\bAd\s*2\b', tail, re.IGNORECASE):
            winner_num = 1
        elif re.search(r'\bAd\s*2\b', tail, re.IGNORECASE) and not re.search(r'\bAd\s*1\b', tail, re.IGNORECASE):
            winner_num = 2
        else:
            print('  [WARNING: could not parse winner]')
            print('  Raw tail:', tail[:200])

    return winner_num, reasoning

print('parse_pairwise_response defined.')

parse_pairwise_response defined.


### B1. Preview Pairwise Prompt

In [12]:
p0   = ads_data[0]
inp0 = input_lookup[p0['asin']]
t0   = list(p0['ads'].keys())[0]
d0   = p0['ads'][t0]
r0   = inp0['topics'][t0]

preview = build_pairwise_prompt(
    product_title    = p0['product_title'],
    description      = inp0['description'],
    features         = inp0['features'],
    topic            = t0,
    positive_reviews = r0['positive_reviews'],
    negative_reviews = r0['negative_reviews'],
    ad1_text         = d0['strategy_a'],
    ad2_text         = d0['strategy_b'],
)
print(preview)

You are an expert advertising evaluator. Two short advertisements for the same product
and feature are shown below. Decide which is MORE PERSUASIVE to a consumer.

PRODUCT: Davison Guitars Full Size Electric Guitar with 10-Watt Amp, Black - Right Handed Beginner Kit with Gig Bag and Accessories
Feature advertised: Tuning stability
Description: If you're searching for the perfect 6-string electric guitar with a full range of accessories for a student, beginner, intermediate, or advanced guitar player, the Davison full-size electric guitar is an outstanding choice. Sold as a complete kit with all of the accessories, this guitar package offers excellent quality at a very attractive price. This guitar from Davison is a 39" solid body electric guitar. It features a flawless glossy finish, a maple neck with a finely finished fretboard, nick
Key features:
- Complete guitar package: This electric guitar kit includes a top-quality Davison electric guitar with a 10W amp, padded gig bag with back

### B2. Run Pairwise (18 calls)

In [13]:
pairwise_results = []
pairwise_raw     = []

random.seed(42)

for product in ads_data:
    asin = product['asin']
    inp  = input_lookup.get(asin, {})
    print(f"\n{'='*60}")
    print(f"{asin} | {product['product_title'][:50]}")

    for topic, ads in product['ads'].items():
        ti   = inp.get('topics', {}).get(topic, {})
        pos  = ti.get('positive_reviews', [])
        neg  = ti.get('negative_reviews', [])
        ad_a = ads['strategy_a']
        ad_b = ads['strategy_b']

        pair_record = {
            'asin':          asin,
            'product_title': product['product_title'],
            'brand':         product['brand'],
            'topic':         topic,
            'ad_a':          ad_a,
            'ad_b':          ad_b,
            'orderings':     [],
        }
        a_wins = 0
        b_wins = 0

        for ordering in ['AB', 'BA']:
            if ordering == 'AB':
                ad1, ad2, label1, label2 = ad_a, ad_b, 'A', 'B'
            else:
                ad1, ad2, label1, label2 = ad_b, ad_a, 'B', 'A'

            print(f'  [{topic[:20]}] ordering={ordering} ...', end=' ')

            prompt = build_pairwise_prompt(
                product_title    = product['product_title'],
                description      = inp.get('description', ''),
                features         = inp.get('features', []),
                topic            = topic,
                positive_reviews = pos,
                negative_reviews = neg,
                ad1_text         = ad1,
                ad2_text         = ad2,
            )
            raw = call_judge(prompt, max_tokens=1500)
            winner_num, reasoning = parse_pairwise_response(raw)

            if winner_num == 1:
                winner_label = label1
            elif winner_num == 2:
                winner_label = label2
            elif winner_num == 'TIE':
                winner_label = 'TIE'
            else:
                winner_label = 'PARSE_ERROR'

            if winner_label == 'A': a_wins += 1
            elif winner_label == 'B': b_wins += 1

            print(f'winner={winner_label}')

            pair_record['orderings'].append({
                'ordering': ordering,
                'winner':   winner_label,
                'reasoning': reasoning,
            })
            pairwise_raw.append({
                'asin': asin, 'topic': topic, 'ordering': ordering,
                'winner': winner_label, 'raw': raw,
            })

            time.sleep(1)

        final_winner = 'A' if a_wins > b_wins else ('B' if b_wins > a_wins else 'TIE')
        pair_record['a_wins']       = a_wins
        pair_record['b_wins']       = b_wins
        pair_record['final_winner'] = final_winner
        pairwise_results.append(pair_record)
        print(f'  => Pair verdict: Strategy {final_winner}  (A_wins={a_wins} B_wins={b_wins})')

print(f'\nPairwise complete. Pairs evaluated: {len(pairwise_results)} / 9')


B002RXXOX8 | Davison Guitars Full Size Electric Guitar with 10-
  [Tuning stability] ordering=AB ... winner=B
  [Tuning stability] ordering=BA ... winner=B
  => Pair verdict: Strategy B  (A_wins=0 B_wins=2)
  [Fret / neck setup] ordering=AB ... winner=B
  [Fret / neck setup] ordering=BA ... winner=B
  => Pair verdict: Strategy B  (A_wins=0 B_wins=2)
  [String quality] ordering=AB ... winner=B
  [String quality] ordering=BA ... winner=B
  => Pair verdict: Strategy B  (A_wins=0 B_wins=2)

B006CYVD5E | Directly Cheap 6 String Acoustic Guitar Pack, Righ
  [Customer service / r] ordering=AB ... winner=B
  [Customer service / r] ordering=BA ... winner=B
  => Pair verdict: Strategy B  (A_wins=0 B_wins=2)
  [String quality] ordering=AB ... winner=A
  [String quality] ordering=BA ... winner=B
  => Pair verdict: Strategy TIE  (A_wins=1 B_wins=1)
  [Tuning stability] ordering=AB ... winner=B
  [Tuning stability] ordering=BA ... winner=B
  => Pair verdict: Strategy B  (A_wins=0 B_wins=2)

B002X49

### B3. Save & Analyse Pairwise

In [14]:
pairwise_rows = []
for r in pairwise_results:
    pairwise_rows.append({
        'asin':          r['asin'],
        'product_title': r['product_title'],
        'brand':         r['brand'],
        'topic':         r['topic'],
        'final_winner':  r['final_winner'],
        'a_wins':        r['a_wins'],
        'b_wins':        r['b_wins'],
        'reasoning_AB':  next((o['reasoning'] for o in r['orderings'] if o['ordering']=='AB'), ''),
        'reasoning_BA':  next((o['reasoning'] for o in r['orderings'] if o['ordering']=='BA'), ''),
    })

pp_df = pd.DataFrame(pairwise_rows)
pp_df.to_csv('scores_pairwise.csv', index=False, encoding='utf-8')

with open('scores_pairwise_raw.json', 'w', encoding='utf-8') as f:
    json.dump(pairwise_raw, f, indent=2, ensure_ascii=False)

print(f'Saved scores_pairwise.csv  ({len(pp_df)} rows)')
print()
print(pp_df[['asin','topic','final_winner','a_wins','b_wins']].to_string(index=False))

Saved scores_pairwise.csv  (9 rows)

      asin                      topic final_winner  a_wins  b_wins
B002RXXOX8           Tuning stability            B       0       2
B002RXXOX8          Fret / neck setup            B       0       2
B002RXXOX8             String quality            B       0       2
B006CYVD5E Customer service / returns            B       0       2
B006CYVD5E             String quality          TIE       1       1
B006CYVD5E           Tuning stability            B       0       2
B002X49732 Customer service / returns            B       0       2
B002X49732             String quality            A       2       0
B002X49732          Fret / neck setup          TIE       1       1


In [15]:
pp_df = pd.read_csv('scores_pairwise.csv')

total   = len(pp_df)
a_total = (pp_df['final_winner'] == 'A').sum()
b_total = (pp_df['final_winner'] == 'B').sum()
t_total = (pp_df['final_winner'] == 'TIE').sum()

print('=== Overall Pairwise Results ===')
print(f'  Strategy A wins : {a_total} / {total}  ({a_total/total*100:.1f}%)')
print(f'  Strategy B wins : {b_total} / {total}  ({b_total/total*100:.1f}%)')
print(f'  Ties            : {t_total} / {total}  ({t_total/total*100:.1f}%)')
print()
primary_winner = 'A' if a_total > b_total else ('B' if b_total > a_total else 'TIE')
print(f'Pairwise primary winner: Strategy {primary_winner}')

print('\n=== By Topic ===')
for topic in pp_df['topic'].unique():
    sub = pp_df[pp_df['topic']==topic]
    a = (sub['final_winner']=='A').sum()
    b = (sub['final_winner']=='B').sum()
    t = (sub['final_winner']=='TIE').sum()
    w = 'A' if a > b else ('B' if b > a else 'tie')
    print(f'  {topic:<35}  A={a}  B={b}  T={t}  -> Strategy {w} leads')

print('\n=== By Product ===')
for asin in pp_df['asin'].unique():
    sub   = pp_df[pp_df['asin']==asin]
    title = sub['product_title'].iloc[0][:45]
    a = (sub['final_winner']=='A').sum()
    b = (sub['final_winner']=='B').sum()
    t = (sub['final_winner']=='TIE').sum()
    w = 'A' if a > b else ('B' if b > a else 'tie')
    print(f'  {title:<45}  A={a}  B={b}  T={t}  -> {w}')

=== Overall Pairwise Results ===
  Strategy A wins : 1 / 9  (11.1%)
  Strategy B wins : 6 / 9  (66.7%)
  Ties            : 2 / 9  (22.2%)

Pairwise primary winner: Strategy B

=== By Topic ===
  Tuning stability                     A=0  B=2  T=0  -> Strategy B leads
  Fret / neck setup                    A=0  B=1  T=1  -> Strategy B leads
  String quality                       A=1  B=1  T=1  -> Strategy tie leads
  Customer service / returns           A=0  B=2  T=0  -> Strategy B leads

=== By Product ===
  Davison Guitars Full Size Electric Guitar wit  A=0  B=3  T=0  -> B
  Directly Cheap 6 String Acoustic Guitar Pack,  A=0  B=2  T=1  -> B
  Crescent MG38-CF 38" Acoustic Guitar Starter   A=1  B=1  T=1  -> tie


### B4. Position Bias Check

In [16]:
with open('scores_pairwise_raw.json') as f:
    raw_data = json.load(f)

ab_res = [r for r in raw_data if r['ordering'] == 'AB']
ba_res = [r for r in raw_data if r['ordering'] == 'BA']

ab_a = sum(1 for r in ab_res if r['winner'] == 'A')
ab_b = sum(1 for r in ab_res if r['winner'] == 'B')
ba_a = sum(1 for r in ba_res if r['winner'] == 'A')
ba_b = sum(1 for r in ba_res if r['winner'] == 'B')

print('=== Position Bias Check ===')
print(f'  AB ordering (A shown first): A wins {ab_a}/9,  B wins {ab_b}/9')
print(f'  BA ordering (B shown first): A wins {ba_a}/9,  B wins {ba_b}/9')

diff = abs(ab_a - ba_a)
if diff == 0:
    print('  -> No position bias detected')
elif diff <= 1:
    print('  -> Minimal position bias (difference <= 1 pair)')
else:
    print(f'  -> Position bias present (diff={diff}) -- note as limitation')

=== Position Bias Check ===
  AB ordering (A shown first): A wins 3/9,  B wins 6/9
  BA ordering (B shown first): A wins 1/9,  B wins 8/9
  -> Position bias present (diff=2) -- note as limitation


---
## Part C: Cross-Method Validation

Compare pointwise composite scores with pairwise win counts.  
Both methods agree -> high-confidence conclusion.  
Methods disagree -> result treated as inconclusive.

In [17]:
pw_df = pd.read_csv('scores_pointwise.csv')
pp_df = pd.read_csv('scores_pairwise.csv')
dims  = ['emotional_resonance', 'argument_strength', 'specificity', 'purchase_motivation']
pw_df['composite'] = pw_df[dims].mean(axis=1)

pw_means  = pw_df.groupby('strategy')['composite'].mean().round(2)
pw_winner = 'A' if pw_means.get('A',0) > pw_means.get('B',0) else 'B'

a_total = (pp_df['final_winner']=='A').sum()
b_total = (pp_df['final_winner']=='B').sum()
pp_winner = 'A' if a_total > b_total else ('B' if b_total > a_total else 'TIE')

print('=== Cross-Method Validation ===')
print(f'  Pointwise  : A={pw_means.get("A","?")}  B={pw_means.get("B","?")}  -> Strategy {pw_winner} wins')
print(f'  Pairwise   : A wins {a_total}/9  B wins {b_total}/9  -> Strategy {pp_winner} wins')
print()
if pw_winner == pp_winner:
    print(f'  AGREEMENT: Both methods favour Strategy {pw_winner}.')
    print(f'  -> High-confidence conclusion: Strategy {pw_winner} produces more persuasive ads.')
else:
    print(f'  DISAGREEMENT: Pointwise favours {pw_winner}, Pairwise favours {pp_winner}.')
    print('  -> Inconclusive. Report both findings and discuss the discrepancy.')

=== Cross-Method Validation ===
  Pointwise  : A=2.69  B=3.17  -> Strategy B wins
  Pairwise   : A wins 1/9  B wins 6/9  -> Strategy B wins

  AGREEMENT: Both methods favour Strategy B.
  -> High-confidence conclusion: Strategy B produces more persuasive ads.


In [18]:
print('=== Topic-Level Cross-Method Comparison ===')
print(f'{"Topic":<35} {"Pointwise":>14} {"Pairwise":>12} {"Agree":>8}')
print('-' * 74)

for topic in pw_df['topic'].unique():
    pw_sub = pw_df[pw_df['topic']==topic].groupby('strategy')['composite'].mean()
    pw_w   = 'A' if pw_sub.get('A',0) > pw_sub.get('B',0) else 'B'
    pw_d   = abs(pw_sub.get('A',0) - pw_sub.get('B',0))

    pp_sub = pp_df[pp_df['topic']==topic]
    pp_a   = (pp_sub['final_winner']=='A').sum()
    pp_b   = (pp_sub['final_winner']=='B').sum()
    pp_w   = 'A' if pp_a > pp_b else ('B' if pp_b > pp_a else 'tie')

    agree   = 'YES' if pw_w == pp_w else 'NO'
    pw_col  = f'Strat {pw_w} (D={pw_d:.2f})'
    pp_col  = f'Strat {pp_w} ({pp_a}-{pp_b})'
    print(f'  {topic:<33} {pw_col:>14} {pp_col:>12} {agree:>8}')

=== Topic-Level Cross-Method Comparison ===
Topic                                    Pointwise     Pairwise    Agree
--------------------------------------------------------------------------
  Tuning stability                  Strat B (D=0.38) Strat B (0-2)      YES
  Fret / neck setup                 Strat B (D=0.50) Strat B (0-1)      YES
  String quality                    Strat B (D=0.17) Strat tie (1-1)       NO
  Customer service / returns        Strat B (D=1.00) Strat B (0-2)      YES


## Save Results Summary

In [19]:
pw_df = pd.read_csv('scores_pointwise.csv')
pp_df = pd.read_csv('scores_pairwise.csv')
dims  = ['emotional_resonance', 'argument_strength', 'specificity', 'purchase_motivation']
pw_df['composite'] = pw_df[dims].mean(axis=1)

pw_means  = pw_df.groupby('strategy')['composite'].mean().round(2)
pw_winner = 'A' if pw_means.get('A',0) > pw_means.get('B',0) else 'B'
a_tot = (pp_df['final_winner']=='A').sum()
b_tot = (pp_df['final_winner']=='B').sum()
t_tot = (pp_df['final_winner']=='TIE').sum()
pp_winner = 'A' if a_tot > b_tot else ('B' if b_tot > a_tot else 'TIE')

lines = []
lines.append('Ad Evaluation -- Full Results Summary')
lines.append('=' * 55)
lines.append(f'Generator : {GEN_MODEL} (OpenAI)')
lines.append(f'Judge     : {JUDGE_MODEL} (DeepSeek)')
lines.append('')

lines.append('--- PART A: POINTWISE (supplementary) ---')
overall = pw_df.groupby('strategy')[dims + ['composite']].mean().round(2)
lines.append(overall.to_string())
lines.append('')
for dim in dims:
    sub = pw_df.groupby('strategy')[dim].mean().round(2)
    w   = 'A' if sub.get('A',0) > sub.get('B',0) else 'B'
    lines.append(f'  {dim}: A={sub.get("A","?")}  B={sub.get("B","?")}  -> {w}')
lines.append('')

lines.append('--- PART B: PAIRWISE (primary) ---')
lines.append(f'  Overall: A wins {a_tot}/9  B wins {b_tot}/9  Ties {t_tot}/9')
lines.append('')
for topic in pp_df['topic'].unique():
    sub = pp_df[pp_df['topic']==topic]
    a = (sub['final_winner']=='A').sum()
    b = (sub['final_winner']=='B').sum()
    t = (sub['final_winner']=='TIE').sum()
    w = 'A' if a > b else ('B' if b > a else 'tie')
    lines.append(f'  {topic:<35}  A={a}  B={b}  T={t}  -> {w}')
lines.append('')

lines.append('--- PART C: CROSS-METHOD VALIDATION ---')
lines.append(f'  Pointwise winner : Strategy {pw_winner}')
lines.append(f'  Pairwise winner  : Strategy {pp_winner}')
agree_str = 'YES -- high confidence' if pw_winner == pp_winner else 'NO -- inconclusive'
lines.append(f'  Agreement        : {agree_str}')

summary = '\n'.join(lines)
with open('results_summary.txt', 'w', encoding='utf-8') as f:
    f.write(summary)
print('Saved results_summary.txt')
print()
print(summary)

Saved results_summary.txt

Ad Evaluation -- Full Results Summary
Generator : gpt-5.4 (OpenAI)
Judge     : deepseek-v4-flash (DeepSeek)

--- PART A: POINTWISE (supplementary) ---
          emotional_resonance  argument_strength  specificity  purchase_motivation  composite
strategy                                                                                     
A                        2.78               2.56         2.56                 2.89       2.69
B                        3.33               3.00         3.00                 3.33       3.17

  emotional_resonance: A=2.78  B=3.33  -> B
  argument_strength: A=2.56  B=3.0  -> B
  specificity: A=2.56  B=3.0  -> B
  purchase_motivation: A=2.89  B=3.33  -> B

--- PART B: PAIRWISE (primary) ---
  Overall: A wins 1/9  B wins 6/9  Ties 2/9

  Tuning stability                     A=0  B=2  -> B
  Fret / neck setup                    A=0  B=1  -> B
  String quality                       A=1  B=1  -> tie
  Customer service / returns        

## Notes on Interpretation

**Theoretical grounding:**
- Pointwise dimensions from ELM (Petty & Cacioppo, 1986) + Rossiter-Percy Grid
- Strategy A = transformational motivation (aspiration, positive emotion)
- Strategy B = informational motivation (problem-removal, reassurance)

**Pairwise design (Zheng et al., 2023):**
- Each pair evaluated twice (AB and BA ordering) to control position bias
- Final winner requires winning both orderings; split orderings -> TIE
- Relative judgements more reliable than absolute scoring

**Cross-company judge:**
- Generator (GPT-5.4, OpenAI) and judge (DeepSeek-V4-Flash, DeepSeek)
- Different companies and model families -> self-enhancement bias strongly mitigated

**Limitations:**
- N=9 pairwise matchups; no statistical significance testing possible at this scale
- LLM judge != real consumers; report as 'under DeepSeek-V4-Flash evaluation'
- All products are budget-tier guitars; findings may not generalise to other categories
- If position bias check shows diff > 1, treat pairwise result with extra caution